In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


# Bhagavad Gita GPT — from scratch

**Goal:** train a GPT-2-style decoder-only Transformer from scratch on the Bhagavad Gita, while retaining the architecture-learning experiments that were useful during implementation.



## 0. Imports and reproducibility

In [1]:
import os
import math
import time
import json
import random
import numpy as np
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cpu
Device: cpu


## 1. Learning / reference: tokenizer and token-ID experiments

The following cells are retained because they explain how tokenization works. They are **not** part of the final training run.

###TOKENIZER (bpe)

> **Environment note:** Install dependencies once in the RunPod terminal if needed: `pip install torch tiktoken matplotlib`.


In [2]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [5]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## 2. Learning / reference: input-target construction

These small examples stay because they explain causal next-token prediction. They are not used by the final training loop.


In [6]:
demo_ids = tokenizer.encode("धर्मो रक्षति रक्षितः", allowed_special={"<|endoftext|>"})
demo_ids = torch.tensor(demo_ids, dtype=torch.long)
context_size_demo = min(8, len(demo_ids) - 1)
x_demo = demo_ids[:context_size_demo]
y_demo = demo_ids[1:context_size_demo + 1]
print("Input IDs :", x_demo.tolist())
print("Target IDs:", y_demo.tolist())
print("Each target token is the next token after the corresponding input token.")


Input IDs : [11976, 100, 11976, 108, 24231, 235, 11976, 106]
Target IDs: [100, 11976, 108, 24231, 235, 11976, 106, 24231]
Each target token is the next token after the corresponding input token.


## 3. Final dataset pipeline

In [7]:
GITA_PATH = "/content/Bhagavad-gita-As-It-Is.txt"

if not os.path.exists(GITA_PATH):
    raise FileNotFoundError(
        f"{GITA_PATH} not found. Put your Bhagavad Gita UTF-8 text file next to this notebook. "
        "Keep the corpus as plain text so the experiment is reproducible."
    )

with open(GITA_PATH, "r", encoding="utf-8") as f:
    text_data = f.read()

print("Characters:", len(text_data))


Characters: 1602995


In [8]:
tokenizer = tiktoken.get_encoding("gpt2")
print("Tokenizer: GPT-2 BPE")
print("Vocabulary size:", tokenizer.n_vocab)


Tokenizer: GPT-2 BPE
Vocabulary size: 50257


In [9]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.input_ids = []
        self.target_ids = []
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length], dtype=torch.long))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1], dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, tokenizer, batch_size, max_length, stride, shuffle, drop_last, num_workers=0):
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers, pin_memory=torch.cuda.is_available())


## 4. Baseline configuration

Start smaller than the original 124M tutorial configuration. This makes repeated experiments affordable while still giving us a meaningful Transformer.

In [10]:
GPT_CONFIG = {
    "vocab_size": tokenizer.n_vocab,
    "context_length": 512,
    "emb_dim": 512,
    "n_heads": 8,
    "n_layers": 8,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

BATCH_SIZE = 8
STRIDE = GPT_CONFIG["context_length"]
TRAIN_RATIO = 0.90

split_idx = int(TRAIN_RATIO * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

train_loader = create_dataloader(train_data, tokenizer, BATCH_SIZE, GPT_CONFIG["context_length"], STRIDE, True, True)
val_loader = create_dataloader(val_data, tokenizer, BATCH_SIZE, GPT_CONFIG["context_length"], STRIDE, False, False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train batches: 108
Validation batches: 13


## 5. Learning / reference: embeddings and attention

These are compact implementation exercises retained from the learning phase. The final GPT model below reuses the same attention implementation.


In [12]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())



In [13]:
demo_vocab = tokenizer.n_vocab
demo_emb_dim = 32
demo_ids = text_to_token_ids("धर्म", tokenizer)
demo_embedding = nn.Embedding(demo_vocab, demo_emb_dim)
token_vecs = demo_embedding(demo_ids)
print("Token embedding shape:", token_vecs.shape)

demo_pos = nn.Embedding(16, demo_emb_dim)
pos_vecs = demo_pos(torch.arange(demo_ids.shape[1]))
print("Position embedding shape:", pos_vecs.shape)
print("Combined shape:", (token_vecs + pos_vecs).shape)


Token embedding shape: torch.Size([1, 8, 32])
Position embedding shape: torch.Size([8, 32])
Combined shape: torch.Size([1, 8, 32])


In [ ]:
# Causal attention: one head, retained as a learning reference.


In [14]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

In [15]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


In [16]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

## 6. Learning / reference: normalization and activation

In [17]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5) #A
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [18]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [19]:
out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Normalized layer outputs:\n", out_norm)
print("Mean:\n", mean)
print("Variance:\n", var)

Normalized layer outputs:
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean:
 tensor([[9.9341e-09],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [20]:
torch.set_printoptions(sci_mode=False)
print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[0.0000],
        [0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [21]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [22]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [23]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )

    def forward(self, x):
        return self.layers(x)

## 7. Final model

In [24]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


In [25]:
model = GPTModel(GPT_CONFIG).to(device)
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {num_params:,}")
print(f"Trainable: {trainable_params:,}")


Parameters: 76,933,120
Trainable: 76,933,120


## 8. Loss and evaluation

In [26]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)
    logits = model(input_batch)
    return nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    if num_batches is None:
        num_batches = len(data_loader)
    num_batches = min(num_batches, len(data_loader))
    total_loss = 0.0
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(data_loader):
            if i >= num_batches:
                break
            total_loss += calc_loss_batch(x, y, model, device).item()
    model.train()
    return total_loss / num_batches


def perplexity(loss):
    return math.exp(loss) if loss < 20 else float("inf")


## 9. Generation

In [27]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1].unsqueeze(-1)
            logits = torch.where(logits < min_val, torch.full_like(logits, float("-inf")), logits)
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and (idx_next == eos_id).all():
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


def generate_and_print_sample(model, tokenizer, device, start_context, max_new_tokens=100, temperature=0.8, top_k=50):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate(model, encoded, max_new_tokens, context_size, temperature, top_k)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.savefig("loss-plot.pdf", bbox_inches="tight")
    plt.show()


epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)


## 10. Training with checkpoint/resume

Checkpoints make the RunPod workflow safe: if the instance stops, training can resume without losing the run.

In [28]:
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(path, model, optimizer, epoch, global_step, tokens_seen, train_losses, val_losses):
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "global_step": global_step,
        "tokens_seen": tokens_seen,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "config": GPT_CONFIG,
    }, path)


def load_checkpoint(path, model, optimizer):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt


def train(model, train_loader, val_loader, optimizer, epochs, eval_every=50, checkpoint_every=500, resume_path=None):
    train_losses, val_losses, tokens_seen = [], [], []
    start_epoch, global_step, seen_tokens = 0, 0, 0

    if resume_path and os.path.exists(resume_path):
        ckpt = load_checkpoint(resume_path, model, optimizer)
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt["global_step"] + 1
        seen_tokens = ckpt["tokens_seen"]
        train_losses = ckpt.get("train_losses", [])
        val_losses = ckpt.get("val_losses", [])
        tokens_seen = ckpt.get("tokens_seen_history", [])
        print(f"Resumed from step {global_step}")

    model.train()
    start_time = time.time()

    for epoch in range(start_epoch, epochs):
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            seen_tokens += input_batch.numel()

            if global_step % eval_every == 0:
                tr = calc_loss_loader(train_loader, model, device, num_batches=min(5, len(train_loader)))
                va = calc_loss_loader(val_loader, model, device, num_batches=min(5, len(val_loader)))
                train_losses.append(tr)
                val_losses.append(va)
                tokens_seen.append(seen_tokens)
                print(f"step={global_step:6d} | train={tr:.4f} (ppl {perplexity(tr):.2f}) | val={va:.4f} (ppl {perplexity(va):.2f})")

            if global_step > 0 and global_step % checkpoint_every == 0:
                path = os.path.join(CHECKPOINT_DIR, f"checkpoint_step_{global_step}.pt")
                save_checkpoint(path, model, optimizer, epoch, global_step, seen_tokens, train_losses, val_losses)
                print("Saved:", path)

            global_step += 1

        prompt = "ॐ"
        encoded = text_to_token_ids(prompt, tokenizer).to(device)
        generated = generate(model, encoded, max_new_tokens=50, context_size=GPT_CONFIG["context_length"], temperature=0.8, top_k=40)
        print("Sample:", token_ids_to_text(generated, tokenizer).replace("\n", " "))

    print(f"Training time: {(time.time() - start_time)/60:.2f} min")
    return train_losses, val_losses, tokens_seen


## 11. Baseline training run

Start conservatively. For the first RunPod smoke test, use a small number of steps/epochs and verify VRAM + tokens/sec before committing to a long run.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)

# First RunPod test: keep this small. Increase after measuring speed and VRAM.
NUM_EPOCHS = 1
EVAL_EVERY = 25
CHECKPOINT_EVERY = 250

train_losses, val_losses, tokens_seen = train(
    model, train_loader, val_loader, optimizer,
    epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    checkpoint_every=CHECKPOINT_EVERY,
)


## 12. Loss curve and experiment record

In [ ]:
import matplotlib.pyplot as plt

if train_losses:
    plt.figure(figsize=(7, 4))
    plt.plot(train_losses, label="Train loss")
    plt.plot(val_losses, label="Validation loss")
    plt.xlabel("Evaluation point")
    plt.ylabel("Cross-entropy loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig("gita_gpt_loss.png", dpi=160)
    plt.show()


## 13. Generation after training

In [ ]:
prompts = [
    "ॐ",
    "धर्म",
    "श्री",
]

model.eval()
for prompt in prompts:
    ids = text_to_token_ids(prompt, tokenizer).to(device)
    out = generate(model, ids, max_new_tokens=100, context_size=GPT_CONFIG["context_length"], temperature=0.8, top_k=40)
    print(f"\nPROMPT: {prompt}\n{token_ids_to_text(out, tokenizer)}")


## 14. Future experiments

Keep this baseline fixed before changing architecture. Next experiments can vary **number of heads, depth/width, context length, tokenizer, RoPE, GQA/MQA, and eventually MoE routing**. Record parameter count, validation loss/perplexity, VRAM, tokens/sec, and generation samples for every run.